# Case Study : Stemming Vs. Lemmitatization in Sentiment Analysis of Movie Reviews

## Objective:

* To compare the effectiveness of Stemming(Using Porter Stemmer)
* Lemmatization(using WordNetLemmatizer) in preprocessing text data
* Sentiment classification, evaluating their impact on model accuracy.

In [66]:
# Import and Load Data

import nltk
import random
import pandas as pd

from nltk.corpus import movie_reviews #Dataset containing labeled movie reviews
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import re

In [67]:
#nltk.download("movie_reviews")
#nltk.download("averaged_perceptron_tagger")
#nltk.download("wordnet")
#nltk.download("omw-1.4")

In [68]:
#movie_reviews

In [69]:
#Load and shuffle 100 positive and 100 negative reviews

docs = [(movie_reviews.raw(fileid), category)
        for category in movie_reviews.categories()
        for fileid in movie_reviews.fileids(category)]

In [70]:
random.shuffle(docs)
docs = docs[:200] #First 200 reviews

#Create DataFrame

df = pd.DataFrame(docs, columns=["review", "label"])
df["label"] = df["label"].map({"pos":1, "neg":0})

In [71]:
df.head()

,review,label
0,one of my brother's favorite movies is h . b ....,0
1,carry on at your convenience is all about the ...,0
2,""" meg ryan is irresistible in the comedy that...",0
3,disney's 35th animated feature-- a retooling o...,1
4,i know there were times during this movie that...,0


In [72]:
df["label"].value_counts()

label
1    103
0     97
Name: count, dtype: int64

## Text Cleanning

In [73]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^\w\s]","",text)

    return text

df["clean"] = df["review"].apply(clean_text)

In [74]:
df.head()

,review,label,clean
0,one of my brother's favorite movies is h . b ....,0,one of my brothers favorite movies is h b ha...
1,carry on at your convenience is all about the ...,0,carry on at your convenience is all about the ...
2,""" meg ryan is irresistible in the comedy that...",0,meg ryan is irresistible in the comedy that ...
3,disney's 35th animated feature-- a retooling o...,1,disneys 35th animated feature a retooling of t...
4,i know there were times during this movie that...,0,i know there were times during this movie that...


## Preprocessing Functions

In [75]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    

In [76]:
# Stopwords

stop_words = set(stopwords.words("english")) - {"not","no","never"}

In [77]:
#Stemming

stemmer = PorterStemmer()

def stem_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    stemmed = [stemmer.stem(w) for w in filtered]
    return " ".join(stemmed)

#stemming process to text


In [78]:
df["stemmed"] = df["clean"].apply(stem_pipeline)

In [79]:
df["stemmed"].head()

0    one brother favorit movi h b halicki 1974 cult...
1    carri conveni goingson factori toilet manufact...
2    meg ryan irresist comedi celebr sisterhood scr...
3    disney 35th anim featur retool olympian legend...
4    know time movi laugh pretti hard problem cant ...
Name: stemmed, dtype: object

## Lemmatization

In [80]:
lemmatizer = WordNetLemmatizer()

def lemma_pipeline(text):
    tokens = word_tokenize(text)
    filtered = [w for w in tokens if w not in stop_words]
    pos_tags = pos_tag(filtered)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in pos_tags]
    return " ".join(lemmatized)

#Define function for lemmatizing process

In [81]:
#nltk.download("averaged_perceptron_tagger_eng")

In [82]:
df["lemmatized"] = df["clean"].apply(lemma_pipeline)

In [83]:
df["lemmatized"].head()

0    one brother favorite movie h b halickis 1974 c...
1    carry convenience goingson factory toilet manu...
2    meg ryan irresistible comedy celebrates sister...
3    disney 35th animated feature retool olympian l...
4    know time movie laugh pretty hard problem cant...
Name: lemmatized, dtype: object

## Classification and Accuracy

In [84]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

#Converts text to numerical TF-IDF feature vectors for machine learnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [85]:
#Prepare data

X_stem = df["stemmed"]
X_lemma = df["lemmatized"]
y = df["label"]

In [86]:
X_stem.head()

0    one brother favorit movi h b halicki 1974 cult...
1    carri conveni goingson factori toilet manufact...
2    meg ryan irresist comedi celebr sisterhood scr...
3    disney 35th anim featur retool olympian legend...
4    know time movi laugh pretti hard problem cant ...
Name: stemmed, dtype: object

In [87]:
X_lemma.head()

0    one brother favorite movie h b halickis 1974 c...
1    carry convenience goingson factory toilet manu...
2    meg ryan irresistible comedy celebrates sister...
3    disney 35th animated feature retool olympian l...
4    know time movie laugh pretty hard problem cant...
Name: lemmatized, dtype: object

In [88]:
y.head()

0    0
1    0
2    0
3    1
4    0
Name: label, dtype: int64

In [89]:
#stemming data split

X_train_s, X_test_s, y_train,y_test = train_test_split(X_stem, y, test_size = 0.2, random_state = 42)

In [90]:
X_train_l, X_test_l, _,_ = train_test_split(X_lemma, y, test_size = 0.2, random_state = 42)

In [91]:
X_train_s.shape

(160,)

In [92]:
y_train.shape

(160,)

In [93]:
vectorizer = TfidfVectorizer()
X_train_s_vec = vectorizer.fit_transform(X_train_s)
X_test_s_vec = vectorizer.transform(X_test_s)

In [94]:
X_train_l_vec = vectorizer.fit_transform(X_train_l)
X_test_l_vec = vectorizer.transform(X_test_l)

In [95]:
#Train and Evaluate

model_s = LogisticRegression(max_iter=1000)
model_s.fit(X_train_s_vec, y_train)

y_pred_s = model_s.predict(X_test_s_vec)

acc_s = accuracy_score(y_test, y_pred_s)

print("Stemming Accuracy:", round(acc_s,2))


Stemming Accuracy: 0.57


In [96]:
#Train and Evaluate

model_l = LogisticRegression(max_iter=1000)
model_l.fit(X_train_l_vec, y_train)

y_pred_l = model_l.predict(X_test_l_vec)

acc_l = accuracy_score(y_test, y_pred_l)

print("Stemming Accuracy:", round(acc_l,2))


Stemming Accuracy: 0.6
